# 05-1. 문자열 정규화 — 풀이 검증

## Goal

필드별 정규화 정책과 원문 보존을 검증한다.

> 학습자용 TODO를 먼저 완성한 뒤 참고한다.


## Setup

fixture와 실행 환경을 확인한다.


In [ ]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("requirements.txt가 있는 저장소 루트에서 JupyterLab을 실행하세요.")


ROOT = find_project_root()
FIXTURE_DIR = ROOT / "fixtures" / "05-text-processing"

assert sys.version_info >= (3, 10)
assert FIXTURE_DIR.is_dir()

print("Python:", sys.version.split()[0])
print("실습 데이터:", FIXTURE_DIR)


import unicodedata
fixture_path = FIXTURE_DIR / "normalization-cases.txt"
cases = [line.split("|", maxsplit=1) for line in fixture_path.read_text(encoding="utf-8").splitlines()]


## Steps

참고 구현을 실행한다.


In [ ]:
def normalize_value(kind: str, raw: str) -> dict:
    if kind == "name":
        normalized = unicodedata.normalize("NFC", raw.strip())
        comparison_key = normalized.casefold()
    elif kind == "email":
        normalized = raw.strip().lower()
        comparison_key = normalized
    elif kind == "description":
        normalized = raw.removesuffix("\n").removesuffix("\r")
        comparison_key = normalized
    else:
        normalized = raw.removesuffix("\n").removesuffix("\r")
        comparison_key = normalized
    return {
        "kind": kind,
        "raw": raw,
        "normalized": normalized,
        "comparison_key": comparison_key,
    }


results = [normalize_value(kind, raw) for kind, raw in cases]
results


## Checks

경계값과 fixture 결과를 대조한다.


In [ ]:
assert normalize_value("email", "  A@EXAMPLE.COM  ")["normalized"] == "a@example.com"
assert normalize_value("path", "/Admin")["normalized"] == "/Admin"
assert normalize_value("name", "  E\u0301lodie  ")["normalized"] == "Élodie"
assert normalize_value("name", "  E\u0301lodie  ")["comparison_key"] == "élodie"
assert all(result["raw"] == raw for result, (_, raw) in zip(results, cases))
print("검증 통과:", len(results), "건")


## Next Steps

정규화 규칙은 데이터 필드의 의미와 일치해야 한다.
